<a href="https://colab.research.google.com/github/flatironinstitute/2026-flatiron-cmb-summer-school/blob/main/notebooks/CMB_School_Simulating_the_CMB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Data Analysis Techniques for Cosmic Microwave Background Maps

### 2026 Flatiron CMB Summer School

This notebook is based primarily on [CMB_School_Part_01.ipynb](https://github.com/jeffmcm1977/CMBAnalysis_SummerSchool/blob/master/CMB_School_Part_01.ipynb) from the [2024 ACT-SPT Analysis School material](https://github.com/jeffmcm1977/CMBAnalysis_SummerSchool/blob/master/) (credits: Jeff McMahon, Renee Hlozek, Sigurd Naess), with further revisions and contributions to the broader series by Alex van Engelen, Alexandra Rahlin, Mathew Madhavacheril, Joshua Kim, Zach Atkins, Will Coulton, Melanie Archipley, and Tom Crawford.
It has been adapted for the [2026 Flatiron CMB Summer School](https://www.simonsfoundation.org/event/cmb-summer-school-2026/) and updated by Susanna Azzoni.

The **Cosmic Microwave Background** (CMB) is the oldest observable light in the universe. As such it carries a wealth of cosmological information including: (1) signals from the early universe (primary anisotropy), and (2) distortions imprinted as this light propagates through the universe and encounters collapsed structures (secondary anisotropy).  Measurements of these signals give us important measurements and constraints on inflationary parameters, dark energy, dark matter, the sum of the neutrino masses, and many astrophysical processes.  The development of CMB instruments and analysis techniques is rapidly evolving.

Modern CMB data analysis is a rapidly evolving field. Modern experiments use large arrays of detectors to map the microwave sky with arcminute-scale resolution and sensitivity to both temperature and polarization. These instruments observe the sky by scanning, producing detector time-streams that must be calibrated, cleaned, filtered, and combined with pointing information to make maps of the microwave sky. Turning these time-ordered data into science-ready maps and spectra is a major computational and statistical challenge: instrumental beams, atmospheric noise, correlated detector noise, masking, filtering, and foreground emission all affect the final cosmological measurements.

Here are example maps from *ACTPol* taken from [Naess et al. 2014](https://arxiv.org/pdf/1405.5524).  Many features are obvious in these maps including: (1) the **primary CMB** visible as waves in the intensity, (2) active galactic nuclei and other bright **astrophysical point sources** which manifest as bright dots, (3) **clusters of galaxies** which show up as darkened point sources.  The figure shows multiple maps; the T is **temperature**, Q and U are **polarization**, and E and B are also polarization but decomposed into a basis such that the E is the amplitude of the curl-free component, and B is the amplitude of the divergence free component of the polarization vector field.

![](http://www.classe.cornell.edu/rsrc/Home/NewsAndEvents/CornellExperimentalCosmologyNews20140528/maps_b.png)

This course introduces the core ideas and practical tools used in modern **CMB data analysis**. CMB maps are not direct images of the early Universe alone: they contain a mixture of cosmological signal, astrophysical foregrounds, instrumental response, and noise. A central goal of CMB analysis is to understand how these different components appear in maps, in Fourier or harmonic space, and in summary statistics such as power spectra, so that robust cosmological and astrophysical information can be extracted.

The course is built around interactive notebooks, supported by complementary lectures. Through simplified but physically motivated examples, we start from simulated CMB skies, including temperature and polarization anisotropies, lensing, point sources, and Sunyaev–Zel’dovich signals. We then add instrumental effects such as beams and noise, and use the resulting maps to introduce common analysis methods: Fourier transforms, filtering, power-spectrum estimation, matched filtering, lensing reconstruction, mapmaking concepts, machine learning with simulations, and cross-correlations with external data sets.

Together, the lectures and notebooks provide a practical entry point into the full CMB analysis pipeline: how maps are simulated, observed, processed, analyzed, and interpreted. The emphasis is on developing physical intuition alongside hands-on experience, so that participants can connect the statistical tools used in CMB analysis to the science goals of current and future experiments, including polarization, B-modes, lensing, galaxy clusters, and analyses with real ACT data.

## Simulating the CMB
We will build a simulated temperature map of the cosmic microwave background (CMB) in three steps:

1. Compute a theory angular power spectrum with CAMB.
2. Convert that one-dimensional spectrum into a two-dimensional flat-sky Fourier-space spectrum.
3. Draw a Gaussian random realization and transform it into a map.

The goal is to understand the minimal pieces that connect a theory spectrum to a map that looks like the sky. This notebook creates the fiducial CAMB spectrum file used by later notebooks, especially `Simulating SZ and Point Sources`, `Instrument Beam, Instrument Noise, and Filtering`, `Power Spectrum Analysis`, and `CMB Polarization`.


## Setup

The notebook can run locally in this repository or on Colab. On Colab, the setup cell installs CAMB if needed and downloads the small helper module used by later notebooks if it is not already present.


In [ ]:
!python -c "import cmb_modules" || ( \
    wget https://raw.githubusercontent.com/flatironinstitute/2026-flatiron-cmb-summer-school/refs/heads/main/notebooks/cmb_modules.py \
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import cmb_modules

## Section 1: The CMB Temperature Power Spectrum

The majority of the information content of the CMB is contained in its angular power spectrum.   This spectrum is the amplitude squared of the magnitude of the temperature fluctuations as a function of $\ell$.  Here $\ell$ is the variable you have seen before with the spherical harmonics (e.g., $Y_{\ell m}$).  $\ell = 0$ corresponds to a constant temperature across the sky, $\ell = 200$ corresponds approximately to scales of $1^\circ$.  For a given set of input cosmological parameters these spectra can be computed with codes including CMBFAST or the more modern equivalent CAMB. 

Recently it has become relatively easy to install and run CAMB through Python.  One can use pip install to do this.


Most of the information in the statistically isotropic CMB temperature field is captured by its *angular power spectrum*. This spectrum is the amplitude squared of the magnitude of the temperature fluctuations as a function of the multipole $\ell$, which labels angular scale. Roughly,
$$
\theta \sim \frac{180^\circ}{\ell}.
$$

Here $\ell$ is the variable you have seen before with the spherical harmonics (e.g., $Y_{\ell m}$). A constant temperature across the sky corresponds to $\ell = 0$, while the first acoustic peak near $\ell \sim 200$ corresponds to degree-scale structure. It is traditional to plot

$$
D_\ell = \frac{\ell(\ell+1)}{2\pi} C_\ell,
$$

because $D_\ell$ shows the contribution to variance per logarithmic interval in $\ell$. To make maps, however, we will convert back to $C_\ell$.

For a given set of input cosmological parameters these spectra can be computed with codes including $\texttt{CMBFAST}$ or the more modern equivalent $\texttt{CAMB}$. $\texttt{CAMB}$ is a numerical code that solves the Boltzmann equations to compute theoretical predictions of cosmological observables — like the CMB angular power spectra for temperature (TT), polarization (EE, BB), and their cross-spectra (TE).

In [ ]:
# Install CAMB
!python -c "import camb" || python -m pip install camb 

Now we can set the cosmological parameters to match the standared cosmological model and ask that the spectrum be computed up to $\ell = 5000$ for our particular case.  We refer you to the [CAMB documentation](https://camb.readthedocs.io/en/latest/) for more details on this package.

In [ ]:
# ============================================================
# 1.1  Compute a fiducial LCDM temperature spectrum with CAMB
# ============================================================

# Import
import camb

# Maximum multipole of the spectra, which is set by the maps resolution
lmax = 5000

# Set up a new set of parameters for CAMB
# The defaults give one massive neutrino and helium set using BBN consistency
cosmo_params = dict(
    H0=67.5, # Hubble constant today, in km/s/Mpc. Sets the present expansion rate.
    ombh2=0.022, # Physical baryon density, controls the baryon content: protons, nuclei, electrons.
    omch2=0.122, # Physical cold dark matter density, controls the CDM abundance.
    mnu=0.06, # Sum of neutrino masses, in eV.
    omk=0.0, # Curvature density, 0.0 means a spatially flat Universe.
    tau=0.06, # Optical depth to reionization.
    As=2.0e-9, # Amplitude of the primordial scalar power spectrum, defined at a pivot scale. Sets the overall amplitude of initial perturbations.
    ns=0.965, # Scalar spectral index. Controls the tilt of the primordial scalar spectrum; ns < 1 means slightly more power on large scales than small scales.
    halofit_version="mead", # Choice of nonlinear matter power spectrum correction. "mead" refers to the Mead/HMCode-style nonlinear prescription.
    lmax=lmax,
)
pars = camb.set_params(**cosmo_params)
pars.set_for_lmax(lmax, lens_potential_accuracy=2)

# Compute results for these parameters
results = camb.get_results(pars)
cls = results.get_total_cls(lmax=lmax, CMB_unit="muK")

# CAMB returns columns TT, EE, BB, TE. Store them with ell as the first column.
# This file will be used by later notebooks.
ell = np.arange(lmax + 1)
np.savetxt("CAMB_fiducial_cosmo_scalCls.dat", np.column_stack([ell, cls]))

print("Saved CAMB_fiducial_cosmo_scalCls.dat with columns ell, TT, EE, BB, TE.")

Note that the “monopole” ($\ell = 0$) and “dipole” ($\ell = 1$) terms are set to zero by default. The monopole is the mean CMB temperature, while the dipole is dominated by our motion with respect to the CMB rest frame; neither is usually treated as part of the intrinsic CMB anisotropy spectrum.

Here is how to read in and plot the CMB temperature spectrum from a CAMB simulation.

In [ ]:
# ============================================================
# 1.2  Read and plot the temperature spectrum
# ============================================================

ell, DlTT = np.loadtxt("CAMB_fiducial_cosmo_scalCls.dat", usecols=(0, 1), unpack=True)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(ell[2:], DlTT[2:], color="C0")
ax.set_xlim(-100, 5000)
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$D_\ell^{TT}$ [$\mu K^2$]")
ax.set_title("Fiducial CMB Temperature Power Spectrum")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


The structure of the temperature power spectrum contains a remarkable amount of cosmological information. At intermediate and high multipoles, corresponding to smaller angular scales, the peaks are produced by **acoustic oscillations in the photon-baryon fluid before recombination**. *Before photons decoupled from matter, gravity pulled matter into overdense regions, while photon pressure pushed back*. This competition set up sound waves in the tightly coupled photon-baryon fluid. At recombination, these waves were effectively frozen into the CMB, leaving a pattern of peaks at characteristic angular scales.

The first acoustic peak, near $\ell \sim 220$ in TT, corresponds roughly to the angular size of the sound horizon at recombination. Its location provided one of the first definitive measurements that the *Universe is close to spatially flat*. The relative heights of the acoustic peaks constrain the baryon density, dark matter density, and other cosmological parameters. For example, changing the baryon density changes the relative heights of the odd and even peaks, while changing the dark matter density affects the expansion history and the gravitational potentials that drive the oscillations.

At even higher multipoles, the spectrum falls off in what is called the **damping tail**. This suppression is caused by *photon diffusion*, often called Silk damping, which smooths away anisotropies on small angular scales before recombination. The damping tail is especially useful for constraining the primordial spectral index from inflation, as well as other parameters that affect the small-scale shape of the spectrum.

At this point, the CMB temperature spectrum is very well measured. Many of the current frontiers are therefore in **polarization** and **secondary anisotropies**, which are not fully represented by the primary temperature spectrum alone. *Polarization* is the faint preferred orientation of the CMB light’s electric field, produced mainly by Thomson scattering in the early Universe. *Secondary anisotropies* are CMB fluctuations generated after recombination, as CMB photons travel from the last-scattering surface to us. *We will talk about these in the upcoming notebooks*. Here, we start by simulating a CMB temperature map.

<font color='red'>EXERCISE 1:</font> As above, you can often find $D_\ell$ plotted instead of $C_\ell$ on the y-axis, where
$D_\ell = \ell (\ell + 1)C_\ell / 2 \pi$, which makes the features more visually distinct. Repeat the above, but also **plotting $C_\ell$ instead of $D_\ell$**. You can try this for different spectra combinations, e.g. TT, TE, EE, BB.

In [ ]:
# Your plot here ...


It is useful, and fun, to vary cosmological parameters in CAMB and see how the spectrum changes. Changing parameters shifts the peak locations, changes their relative amplitudes, and modifies the damping tail. This is one reason the CMB temperature spectrum has been so powerful: a wealth of cosmological parameters, including the baryon density, dark matter density, curvature, dark energy, and the properties of the primordial fluctuations, are constrained by measurements of this spectrum.

<font color='red'>EXERCISE 2:</font> **Change one cosmological parameter** below, such as `H0`, `ombh2`, `omch2`, or `ns`. Which part of the spectrum responds most strongly? Note: save the spectra with to a different file to avoid overwriting `CAMB_fiducial_cosmo_scalCls.dat`, and read it in with a different name e.g. `DlTT_modified`.


In [ ]:
# Your code/plot here ...


## Section 2: Simulating a Temperature Anisotropy Map

In this step we generate a simulated map of the CMB sky with the spectrum we read in above.  Since the power spectrum is a function of $\ell$ we need to do much of the work in harmonic space.  If we were generating a map on the full sky we would need to work with spherical harmonics.  Here we consider a small patch of sky $(\sim 10^\circ \times 10^\circ)$ where we can use the "flat-sky" approximation and replace $\ell$ with $k = \sqrt{k_x^2 + k_y^2}$.  There is a linear dependence between these variables defined by $\ell = k* 2 \pi$.

The simulation recipe is:

1. Convert $D_\ell^{TT}$ to $C_\ell^{TT}$.
2. Build a two-dimensional version of $C_\ell^{TT}$ on a Fourier grid.
3. Generate a Gaussian random field in Fourier space.
4. Weight each Fourier mode by $\sqrt{C_\ell^{TT}}$.
5. Fourier transform back to real space.


In [ ]:
# ============================================================
# 2.1  Map geometry and plotting helpers
# ============================================================

N = 2**10              # pixels on a side; powers of 2 keep FFTs fast
pix_size = 0.5         # pixel size in arcminutes
X_width = N * pix_size / 60.0
Y_width = N * pix_size / 60.0
c_min, c_max = -400, 400


def plot_cmb_map(map_to_plot, title=None, vmin=c_min, vmax=c_max, cmap="RdBu_r"):
    print(f"map mean: {np.mean(map_to_plot): .3f} muK")
    print(f"map rms:  {np.std(map_to_plot): .3f} muK")

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        map_to_plot,
        interpolation="bilinear",
        origin="lower",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=[0, X_width, 0, Y_width],
    )
    ax.set_xlabel(r"Angle [deg]")
    ax.set_ylabel(r"Angle [deg]")
    if title is not None:
        ax.set_title(title)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax, label="Temperature [muK]")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 2.2  CMB temperature map simulator
# ============================================================

def make_cmb_temperature_map(N, pix_size, ell, DlTT, rng=None, return_steps=False):
    # If no random-number generator is supplied, create one.
    # This makes the function self-contained, but still allows reproducibility
    # if the user passes rng = np.random.default_rng(seed).
    if rng is None:
        rng = np.random.default_rng()

    # Convert inputs to NumPy arrays in case they were passed as lists.
    ell = np.asarray(ell)
    DlTT = np.asarray(DlTT)

    # Convert from D_ell to C_ell.
    #     C_ell = D_ell * 2 pi / [ell(ell+1)]
    # We only do this for ell >= 2 to avoid division by zero for ell = 0
    # and to avoid the dipole ell = 1
    ClTT = np.zeros_like(DlTT, dtype=float)
    valid = ell >= 2
    ClTT[valid] = DlTT[valid] * 2.0 * np.pi / (ell[valid] * (ell[valid] + 1.0))

    # Build dimensionless coordinates across the square patch.
    # inds runs from -0.5 to +0.5, so X and Y describe positions across
    # the map in normalized units, not yet in physical angular units.
    inds = np.linspace(-0.5, 0.5, N)
    X, Y = np.meshgrid(inds, inds)

    # Radius from the center of the 2D grid.
    # This will be used as a simple radial Fourier-space coordinate.
    R = np.sqrt(X**2 + Y**2)

    # Convert pixel size from arcmin to radians.
    pix_to_rad = pix_size / 60.0 * np.pi / 180.0

    # Approximate conversion from grid radius to multipole ell.
    # Smaller pixels correspond to access to larger ell.
    # The factor 2 pi / pixel_size is the characteristic Fourier scale
    # associated with one pixel.
    ell_scale_factor = 2.0 * np.pi / pix_to_rad

    # Make a 2D ell grid.
    # Each Fourier-space pixel is assigned an approximate multipole ell.
    ell2d = R * ell_scale_factor

    # Make a lookup table for C_ell values.
    # ClTT is only defined at the ell values provided by the input spectrum.
    # ClTT_expanded is an array that can be indexed by integer ell values.
    ClTT_expanded = np.zeros(int(ell2d.max()) + 1)

    # Copy as many C_ell values as possible into the lookup table.
    # This protects against cases where ell2d extends beyond the input ell range.
    ncopy = min(ClTT_expanded.size, ClTT.size)
    ClTT_expanded[:ncopy] = ClTT[:ncopy]

    # Assign a C_ell value to every 2D Fourier-space pixel.
    # ell2d is continuous, so we convert it to integers before indexing.
    # This means each Fourier pixel gets the C_ell corresponding to its
    # nearest lower integer ell.
    ClTT2d = ClTT_expanded[ell2d.astype(int)]

    # Start from a white-noise Gaussian random map.
    # This map has independent Gaussian random values in each pixel.
    # It has no CMB-like correlations yet.
    random_map = rng.normal(0.0, 1.0, size=(N, N))

    # Fourier transform the white-noise map.
    # In Fourier space, we can impose the desired CMB power spectrum
    # mode by mode.
    ft_random_map = np.fft.fft2(random_map)

    # Weight each Fourier mode by sqrt(C_ell).
    # Since power is proportional to amplitude squared, multiplying the
    # Fourier amplitudes by sqrt(C_ell) gives the map approximately the
    # desired power spectrum C_ell.
    ft_weighted = np.sqrt(ClTT2d) * ft_random_map

    # Transform back to map space.
    # fftshift is used here to match the convention of the constructed
    # radial ell grid, where ell = 0 is placed at the center of the array.
    cmb_t = np.fft.ifft2(np.fft.fftshift(ft_weighted))

    # The inverse FFT returns a complex array because of numerical details.
    # The imaginary part should be negligible, so we keep the real part.
    # The division by pix_to_rad is a normalization factor used in this
    # simple flat-sky convention.
    cmb_t = np.real(cmb_t) / pix_to_rad

    # Optionally return intermediate objects so students can inspect them.
    if return_steps:
        return cmb_t, ell2d, ClTT2d, ft_weighted

    # Otherwise return only the final simulated CMB temperature map.
    return cmb_t

In [ ]:
# ============================================================
# 2.3  Make and plot one CMB realization
# ============================================================

# Random number generator used to make the Gaussian realization.
rng = np.random.default_rng(2026)

# Make one flat-sky Gaussian CMB temperature realization
CMB_T, ell2d, ClTT2d, ft_weighted = make_cmb_temperature_map(
    N, pix_size, ell, DlTT, rng=rng, return_steps=True
)

# Plot
plot_cmb_map(CMB_T, title="One Simulated CMB Temperature Map")


This plot shows simulated CMB map we just generated. If you rerun the previous cell with a different random seed, the hot and cold spots move around – i.e. the spectrum fixes the statistical pattern, not the exact arrangement of the spots in our one universe. However, you will see that the typical size of the brightest and darkest spots will stay around  1$^\circ$, corresponding to the peak of the angular power spectrum. 

In [ ]:
# ============================================
# 2.4  Look at the Fourier-space ingredients
# ============================================

from matplotlib.patches import Circle

# Helper: average a 2D Fourier-space quantity in annuli of ell
def radial_bin_mean(ell_grid, quantity_grid, bins):
    ell_flat = ell_grid.ravel()
    q_flat = quantity_grid.ravel()

    which_bin = np.digitize(ell_flat, bins) - 1
    ell_mid = 0.5 * (bins[:-1] + bins[1:])

    q_mean = np.full(len(ell_mid), np.nan)

    for i in range(len(ell_mid)):
        m = (which_bin == i) & np.isfinite(q_flat)
        if np.any(m):
            q_mean[i] = np.mean(q_flat[m])

    return ell_mid, q_mean
    
# ------------------------------------------------------------
# Part A: 1D view
# ------------------------------------------------------------
# The Fourier realization is random mode by mode.
# To see the connection to the input C_ell, average modes in rings of constant ell.

power_realization = np.abs(ft_weighted)**2

# In this toy FFT convention, divide by N^2 to put the random Fourier power
# roughly on the same scale as C_ell. We will also rescale below for display.
Cl_realization_2d = power_realization / N**2

# Convert both the input and one realization from C_ell to D_ell,
# because D_ell is the quantity students usually recognize.
Dl_input_2d = ell2d * (ell2d + 1.0) * ClTT2d / (2.0 * np.pi)
Dl_realization_2d = ell2d * (ell2d + 1.0) * Cl_realization_2d / (2.0 * np.pi)

ell_max_for_bins = min(np.max(ell), np.max(ell2d))
bins = np.linspace(2, ell_max_for_bins, 80)

ell_bin, Dl_input_bin = radial_bin_mean(ell2d, Dl_input_2d, bins)
_, Dl_realization_bin = radial_bin_mean(ell2d, Dl_realization_2d, bins)

# The precise normalization depends on FFT conventions in this toy simulator.
# For teaching, rescale the realization so we can compare the shape clearly.
good = (
    np.isfinite(Dl_input_bin)
    & np.isfinite(Dl_realization_bin)
    & (Dl_input_bin > 0)
    & (Dl_realization_bin > 0)
    & (ell_bin > 50)
    & (ell_bin < min(3000, ell_max_for_bins))
)

display_scale = np.nanmedian(Dl_input_bin[good] / Dl_realization_bin[good])
Dl_realization_bin_display = Dl_realization_bin * display_scale


fig, ax = plt.subplots(figsize=(7, 4.8))

ax.plot(ell[2:], DlTT[2:], lw=2, label="Input theory")
ax.scatter(
    ell_bin,
    Dl_realization_bin_display,
    s=18,
    alpha=0.75,
    label="One random realization, averaged in rings",
)

ax.set_xlim(2, min(5000, ell_max_for_bins))
ax.set_ylim(1, 1.2 * np.nanmax(DlTT[(ell > 2) & (ell < min(5000, ell_max_for_bins))]))
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$D_\ell^{TT}$ [$\mu K^2$]")
ax.set_title("A random map follows the input spectrum only after averaging modes")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

This plot compares the input theoretical CMB temperature power spectrum with the power spectrum recovered from one random simulated map.
The smooth curve is the input theory, $D_\ell^{TT}$. This is the spectrum we want the simulated map to follow on average. The points come from the random Fourier realization used to make the map. Individual Fourier modes are random, so they do not exactly equal the theory mode by mode. However, after averaging many modes with similar $\ell$, the realization follows the same overall shape as the input spectrum.

In [ ]:
# ------------------------------------------------------------
# Part B: 2D view
# ------------------------------------------------------------
# Note: ClTT2d and ft_weighted are already in a centered convention:
# ell = 0 is at the center of the image.

def log10_for_display(x):
    """Take log10 while avoiding -inf from exact zeros."""
    x = np.asarray(x)
    positive = x[x > 0]

    if positive.size == 0:
        floor = 1.0e-30
    else:
        floor = np.nanpercentile(positive, 1)

    return np.log10(np.maximum(x, floor))

# Convert the pixel size from arcminutes to radians.
pix_to_rad = pix_size / 60.0 * np.pi / 180.0

# The Fourier variable corresponding to an angle theta is roughly ell ~ 2 pi / theta.
# This sets the maximum scale of the ell-axis for the Fourier grid.
ell_scale_factor = 2.0 * np.pi / pix_to_rad

# Build a 1D Fourier-axis for plotting, running from negative to positive ell values.
ell_axis = np.linspace(-0.5, 0.5, N) * ell_scale_factor
extent = [ell_axis[0], ell_axis[-1], ell_axis[0], ell_axis[-1]]

# Crop to the useful part of Fourier space.
# The full grid extends far beyond the maximum ell in the input CAMB spectrum,
# so most of the full image is exactly zero and visually uninformative.
ell_crop = min(5200, ell_max_for_bins)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# Left: smooth input recipe
im0 = axes[0].imshow(
    log10_for_display(ClTT2d),
    origin="lower",
    cmap="magma",
    extent=extent,
)
axes[0].set_title(r"Input recipe: $\log_{10} C_\ell^{TT}$")

# Right: one noisy random draw from that recipe
im1 = axes[1].imshow(log10_for_display(power_realization),origin="lower",cmap="magma",extent=extent)
axes[1].set_title(r"One draw: $\log_{10} |\tilde T(\ell_x,\ell_y)|^2$")

for ax in axes:
    ax.set_xlim(-ell_crop, ell_crop)
    ax.set_ylim(-ell_crop, ell_crop)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$\ell_x$")
    ax.set_ylabel(r"$\ell_y$")

    # Mark ell = 0.
    ax.plot(0, 0, marker="+", markersize=10, color="white", mew=2)
    ax.text(120, 120, r"$\ell=0$", color="white", fontsize=9)

    # Draw rings of constant ell.
    for ring_ell in [200, 1000, 2000, 4000]:
        if ring_ell < ell_crop:
            circ = Circle((0, 0), ring_ell, fill=False, color="white", lw=0.8, alpha=0.55,)
            ax.add_patch(circ)

axes[0].text(0.03,0.96,"Smooth: expected variance per Fourier mode",transform=axes[0].transAxes,ha="left",va="top",color="white",fontsize=9,)

axes[1].text(0.03,0.96,"Noisy: one random Gaussian realization",transform=axes[1].transAxes,ha="left",va="top",color="white",fontsize=9,)

fig.colorbar(im0, ax=axes[0], shrink=0.85, label=r"$\log_{10} C_\ell^{TT}$")
fig.colorbar(im1, ax=axes[1], shrink=0.85, label=r"$\log_{10} |\tilde T|^2$")

plt.show()

Each pixel is a Fourier mode labelled by $(\ell_x, \ell_y)$.

The left panel shows the theoretical $C_\ell^{TT}$ copied onto a 2D Fourier grid. Distance from the centre corresponds to angular multipole $\ell = \sqrt{\ell_x^2 + \ell_y^2}$, so rings correspond to modes with the same angular scale. The image is the smooth input recipe for the simulation, i.e. it shows how much power each Fourier mode should have before drawing a random realization.

The right panel shows one noisy random draw of the Fourier modes. The smooth theory $C_\ell^{TT}$ sets the variance, but the actual amplitudes fluctuate randomly. After inverse Fourier transforming these modes, we obtain one simulated CMB temperature map.

<font color='red'>EXERCISE 3:</font> **Make several random CMB skies**. So far we have made one simulated CMB temperature map from an input theory power spectrum.  But the CMB sky is a random field: the theory tells us the statistical distribution, not the exact hot and cold spots. In this exercise, repeat the simulation using several different random seeds.

1. Generate several independent Fourier-space realizations using the same input \(C_\ell^{TT}\), but different random seeds.
2. Inverse Fourier transform each realization to make a simulated temperature map.
3. Plot the maps side by side. Notice that the hot and cold spots move around from realization to realization.
4. For each realization, estimate the binned 1D power spectrum and overlay the result on the input theory curve.
5. Finally, plot the 2D Fourier power spectrum for each realization.

Questions to think about: Do the maps look identical? Do they have the same typical angular scale of hot and cold spots? Does each binned power spectrum exactly match the theory curve? What improves when you average over more modes or more realizations?

In [ ]:
# Your code/plots here ...


## Section 3: What Angular Scales Make the Map?

The map is easier to understand if we split the spectrum into broad bands of $\ell$. Low-$\ell$ modes are large angular scales; high-$\ell$ modes are small angular scales. We can zero out part of the input spectrum and simulate again.


<font color='red'>EXERCISE 4:</font> Now we know how to go from an assumed cosmology, to a power spectra, to a map. We want to understand: *which multipoles make which structures*? Separate the input CMB temperature power spectrum into two pieces: 1) **larger angular scales**, using only modes with $\ell < 700$ and 2) **smaller angular scales**, using only modes with with $\ell < 700$. Move the split from $\ell=700$ to another value. For example, what happens near the first acoustic peak at $\ell \sim 200$? What happens if you keep only $\ell > 2000$?

In [ ]:
# Your code/plots here ...


## Section 4: The power spectrum tells us about the Universe

We have seen how a CMB map shows one particular realization of the microwave sky: the hot and cold spots we observe in our Universe. The exact positions of those spots are random. The deeper cosmological information is in their statistical pattern.

We have also seen that the temperature power spectrum, $C_\ell^{TT}$, measures how much fluctuation power there is as a function of angular scale. Low multipoles $\ell$ describe large angular scales. High multipoles describe small angular scales.

This is why the CMB power spectrum is such a powerful cosmological observable. The locations of the acoustic peaks tell us about the geometry of the Universe and the sound horizon at recombination. The relative heights of the peaks tell us about the amounts of baryons and dark matter. The damping tail at high $\ell$ tells us about photon diffusion in the early Universe and the primordial fluctuation spectrum.

Changing the cosmological model changes the predicted $C_\ell$. Changing $C_\ell$ changes the statistical appearance of simulated maps. Schematically:

$$
\text{cosmological parameters}
\quad \longrightarrow \quad
C_\ell
\quad \longrightarrow \quad
\text{simulated CMB maps}.
$$

**Real CMB analysis runs this logic in reverse. We observe maps of the microwave sky, estimate their spectra, and use those spectra to infer the contents, geometry, and history of the Universe.**

<font color='red'>EXERCISE 5: </font> Re-run your code with the spectrum from your strange model univese. Compare how the maps look. Note: use the same random seed for each map. This means that the differences between panels come mainly from changing the input spectrum, not from drawing a completely different random sky.

In [ ]:
# Your code/plots here ...
